# PATIENT DATA

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


PATIENT_SOURCE = "patient_kg_dev.staged.patients"
PATIENT_TARGET = "patient_kg_dev.silver.patients"

patients_source = spark.table(PATIENT_SOURCE)

print(f"Source rows: {patients_source.count()}")

**Normalize the patients**

In [0]:
money_type = DecimalType(18, 2)

patients_normalized = patients_source.select(
    F.col("Id").alias("patient_id"),

    F.col("BIRTHDATE").alias("birth_date_source"),
    F.to_date(
        F.col("BIRTHDATE"),
        "yyyy-MM-dd",
    ).alias("birth_date"),

    F.col("DEATHDATE").alias("death_date_source"),
    F.when(
        F.col("DEATHDATE").isNull()
        | (F.trim(F.col("DEATHDATE")) == ""),
        F.lit(None).cast("date"),
    ).otherwise(
        F.to_date(F.col("DEATHDATE"), "yyyy-MM-dd")
    ).alias("death_date"),

    F.col("GENDER").alias("gender"),
    F.col("RACE").alias("race"),
    F.col("ETHNICITY").alias("ethnicity"),
    F.col("CITY").alias("city"),
    F.col("STATE").alias("state"),

    F.col("HEALTHCARE_EXPENSES").alias(
        "healthcare_expenses_source"
    ),
    F.col("HEALTHCARE_EXPENSES")
    .cast(money_type)
    .alias("healthcare_expenses"),

    F.col("HEALTHCARE_COVERAGE").alias(
        "healthcare_coverage_source"
    ),
    F.col("HEALTHCARE_COVERAGE")
    .cast(money_type)
    .alias("healthcare_coverage"),

    F.col("INCOME").alias("income_source"),
    F.col("INCOME")
    .cast(money_type)
    .alias("income"),

    F.col("_source_file"),
    F.col("_source_file_path"),
    F.col("_row_content_sha256"),

    F.lit("patients_v1").alias(
        "_normalization_version"
    ),
    F.current_timestamp().alias("_normalized_at"),
)

**Patient identifier validation: **

In [0]:
missing_patient_ids = (
    patients_normalized
    .filter(
        F.col("patient_id").isNull()
        | (F.trim(F.col("patient_id")) == "")
    )
    .count()
)

duplicate_patient_ids = (
    patients_normalized
    .groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Missing patient IDs: {missing_patient_ids}")
print(f"Duplicate patient IDs: {duplicate_patient_ids}")

**Date parsing validation**

In [0]:
invalid_birth_dates = (
    patients_normalized
    .filter(
        F.col("birth_date_source").isNotNull()
        & (F.trim(F.col("birth_date_source")) != "")
        & F.col("birth_date").isNull()
    )
    .count()
)

invalid_death_dates = (
    patients_normalized
    .filter(
        F.col("death_date_source").isNotNull()
        & (F.trim(F.col("death_date_source")) != "")
        & F.col("death_date").isNull()
    )
    .count()
)

print(f"Invalid birth dates: {invalid_birth_dates}")
print(f"Invalid death dates: {invalid_death_dates}")

**Numeric parsing validation**

In [0]:
numeric_fields = [
    ("healthcare_expenses_source", "healthcare_expenses"),
    ("healthcare_coverage_source", "healthcare_coverage"),
    ("income_source", "income"),
]

for source_column, normalized_column in numeric_fields:
    invalid_count = (
        patients_normalized
        .filter(
            F.col(source_column).isNotNull()
            & (F.trim(F.col(source_column)) != "")
            & F.col(normalized_column).isNull()
        )
        .count()
    )

    print(
        f"{normalized_column} parsing failures: "
        f"{invalid_count}"
    )

**Row-count reconciliation**

In [0]:
source_count = patients_source.count()
normalized_count = patients_normalized.count()

if source_count != normalized_count:
    raise RuntimeError(
        f"Patient normalization changed row count: "
        f"source={source_count}, "
        f"normalized={normalized_count}"
    )

**Write the Silver table**

In [0]:
(
    patients_normalized.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PATIENT_TARGET)
)

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT patient_id) AS distinct_patients,
    SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END)
        AS missing_birth_dates,
    SUM(CASE
        WHEN _row_content_sha256 IS NULL THEN 1
        ELSE 0
    END) AS missing_lineage
FROM patient_kg_dev.silver.patients;

In [0]:
%sql
SELECT
    patient_id,
    birth_date_source,
    birth_date,
    death_date_source,
    death_date,
    income_source,
    income,
    _normalization_version
FROM patient_kg_dev.silver.patients
ORDER BY patient_id;

# ENCOUNTER DATA

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


ENCOUNTER_SOURCE = "patient_kg_dev.staged.encounters"
ENCOUNTER_TARGET = "patient_kg_dev.silver.encounters"

encounters_source = spark.table(ENCOUNTER_SOURCE)

print(f"Staged encounters: {encounters_source.count()}")

**Load the staged encounters**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


ENCOUNTER_SOURCE = "patient_kg_dev.staged.encounters"
ENCOUNTER_TARGET = "patient_kg_dev.silver.encounters"

encounters_source = spark.table(ENCOUNTER_SOURCE)

print(f"Staged encounters: {encounters_source.count()}")

**Normalize the encounters:**

In [0]:
money_type = DecimalType(18, 2)
timestamp_format = "yyyy-MM-dd'T'HH:mm:ssX"

encounters_normalized = encounters_source.select(
    F.col("Id").alias("encounter_id"),
    F.col("PATIENT").alias("patient_id"),

    F.col("START").alias("start_at_source"),
    F.to_timestamp(
        F.col("START"),
        timestamp_format,
    ).alias("start_at"),

    F.col("STOP").alias("stop_at_source"),
    F.to_timestamp(
        F.col("STOP"),
        timestamp_format,
    ).alias("stop_at"),

    F.col("ORGANIZATION").alias("organization_id"),
    F.col("PROVIDER").alias("provider_id"),
    F.col("PAYER").alias("payer_id"),

    F.col("ENCOUNTERCLASS").alias("encounter_class"),
    F.col("CODE").alias("code"),
    F.col("DESCRIPTION").alias("source_description"),

    F.col("BASE_ENCOUNTER_COST").alias(
        "base_cost_source"
    ),
    F.col("BASE_ENCOUNTER_COST")
    .cast(money_type)
    .alias("base_cost"),

    F.col("TOTAL_CLAIM_COST").alias(
        "total_claim_cost_source"
    ),
    F.col("TOTAL_CLAIM_COST")
    .cast(money_type)
    .alias("total_claim_cost"),

    F.col("PAYER_COVERAGE").alias(
        "payer_coverage_source"
    ),
    F.col("PAYER_COVERAGE")
    .cast(money_type)
    .alias("payer_coverage"),

    F.when(
        F.col("REASONCODE").isNull()
        | (F.trim(F.col("REASONCODE")) == ""),
        F.lit(None).cast("string"),
    ).otherwise(
        F.col("REASONCODE")
    ).alias("reason_code"),

    F.col("REASONDESCRIPTION").alias(
        "reason_description_source"
    ),

    F.lit(None).cast("string").alias("code_system"),
    F.lit("not_supplied").alias(
        "code_system_provenance"
    ),

    F.col("_source_file"),
    F.col("_source_file_path"),
    F.col("_row_content_sha256"),

    F.lit("encounters_v1").alias(
        "_normalization_version"
    ),
    F.current_timestamp().alias("_normalized_at"),
)

**Validate identifiers**

In [0]:
missing_encounter_ids = (
    encounters_normalized
    .filter(
        F.col("encounter_id").isNull()
        | (F.trim(F.col("encounter_id")) == "")
    )
    .count()
)

duplicate_encounter_ids = (
    encounters_normalized
    .groupBy("encounter_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_patient_ids = (
    encounters_normalized
    .filter(
        F.col("patient_id").isNull()
        | (F.trim(F.col("patient_id")) == "")
    )
    .count()
)

print(f"Missing encounter IDs: {missing_encounter_ids}")
print(f"Duplicate encounter IDs: {duplicate_encounter_ids}")
print(f"Missing patient IDs: {missing_patient_ids}")

**Validate patient references**

In [0]:
silver_patient_ids = (
    spark.table("patient_kg_dev.silver.patients")
    .select("patient_id")
    .distinct()
)

orphan_encounters = (
    encounters_normalized
    .select("encounter_id", "patient_id")
    .join(
        silver_patient_ids,
        on="patient_id",
        how="left_anti",
    )
)

orphan_encounter_count = orphan_encounters.count()

print(
    f"Encounters referencing missing patients: "
    f"{orphan_encounter_count}"
)

**Validate timestamp parsing:**

In [0]:
invalid_start_timestamps = (
    encounters_normalized
    .filter(
        F.col("start_at_source").isNotNull()
        & (F.trim(F.col("start_at_source")) != "")
        & F.col("start_at").isNull()
    )
    .count()
)

invalid_stop_timestamps = (
    encounters_normalized
    .filter(
        F.col("stop_at_source").isNotNull()
        & (F.trim(F.col("stop_at_source")) != "")
        & F.col("stop_at").isNull()
    )
    .count()
)

print(f"Invalid starts: {invalid_start_timestamps}")
print(f"Invalid stops: {invalid_stop_timestamps}")

**Validate encounter intervals:**

In [0]:
invalid_intervals = (
    encounters_normalized
    .filter(
        F.col("start_at").isNotNull()
        & F.col("stop_at").isNotNull()
        & (F.col("start_at") > F.col("stop_at"))
    )
)

invalid_interval_count = invalid_intervals.count()

print(f"START after STOP: {invalid_interval_count}")

**Validate financial parsing:**

In [0]:
financial_fields = [
    ("base_cost_source", "base_cost"),
    ("total_claim_cost_source", "total_claim_cost"),
    ("payer_coverage_source", "payer_coverage"),
]

for source_column, normalized_column in financial_fields:
    failures = (
        encounters_normalized
        .filter(
            F.col(source_column).isNotNull()
            & (F.trim(F.col(source_column)) != "")
            & F.col(normalized_column).isNull()
        )
        .count()
    )

    print(f"{normalized_column} failures: {failures}")

**Check reason-code consistency:**

In [0]:
incomplete_reason_pairs = (
    encounters_normalized
    .filter(
        (
            F.col("reason_code").isNull()
            & F.col("reason_description_source").isNotNull()
            & (
                F.trim(
                    F.col("reason_description_source")
                ) != ""
            )
        )
        |
        (
            F.col("reason_code").isNotNull()
            & (
                F.col("reason_description_source").isNull()
                | (
                    F.trim(
                        F.col("reason_description_source")
                    ) == ""
                )
            )
        )
    )
    .count()
)

print(
    f"Incomplete reason code/description pairs: "
    f"{incomplete_reason_pairs}"
)

**Reconcile and write**

In [0]:
source_count = encounters_source.count()
normalized_count = encounters_normalized.count()

if source_count != normalized_count:
    raise RuntimeError(
        f"Encounter normalization changed row count: "
        f"source={source_count}, "
        f"normalized={normalized_count}"
    )

if missing_encounter_ids > 0:
    raise RuntimeError("Encounter IDs are missing")

if duplicate_encounter_ids > 0:
    raise RuntimeError("Encounter IDs are duplicated")

if missing_patient_ids > 0:
    raise RuntimeError("Encounter patient IDs are missing")

if orphan_encounter_count > 0:
    raise RuntimeError(
        "Some encounters reference missing patients"
    )

if invalid_start_timestamps > 0:
    raise RuntimeError(
        "Some encounter start timestamps failed parsing"
    )

if invalid_stop_timestamps > 0:
    raise RuntimeError(
        "Some encounter stop timestamps failed parsing"
    )

if invalid_interval_count > 0:
    raise RuntimeError(
        "Some encounters have START after STOP"
    )

In [0]:
(
    encounters_normalized.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ENCOUNTER_TARGET)
)

**Final SQL verification**

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT encounter_id)
        AS distinct_encounters,
    COUNT(DISTINCT patient_id)
        AS distinct_patients,
    SUM(CASE WHEN start_at IS NULL THEN 1 ELSE 0 END)
        AS missing_starts,
    SUM(CASE WHEN stop_at IS NULL THEN 1 ELSE 0 END)
        AS missing_stops,
    SUM(CASE WHEN start_at > stop_at THEN 1 ELSE 0 END)
        AS invalid_intervals,
    SUM(CASE
        WHEN _row_content_sha256 IS NULL THEN 1
        ELSE 0
    END) AS missing_lineage
FROM patient_kg_dev.silver.encounters;

In [0]:
from pyspark.sql import functions as F

CATALOG = "patient_kg_dev"
STAGED = f"{CATALOG}.staged"
SILVER = f"{CATALOG}.silver"

NORMALIZATION_VERSION = "v1"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:
def blank_to_null(column_name):
    return F.when(
        F.trim(F.col(column_name)) == "",
        None
    ).otherwise(F.trim(F.col(column_name)))


def source_lineage_columns():
    return [
        F.col("_source_file"),
        F.col("_source_file_path"),
        F.col("_source_file_size"),
        F.col("_source_file_modified_at"),
        F.col("_row_content_sha256"),
        F.col("_ingestion_run_id"),
        F.col("_ingested_at"),
        F.lit(NORMALIZATION_VERSION).alias("_normalization_version"),
        F.current_timestamp().alias("_normalized_at")
    ]

**ENCOUNTERS**

In [0]:
encounters = (
    spark.table(f"{STAGED}.encounters")
    .select(
        blank_to_null("Id").alias("encounter_id"),
        blank_to_null("PATIENT").alias("patient_id"),

        F.col("START").alias("start_at_source"),
        F.to_timestamp(
            "START", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("start_at"),

        F.col("STOP").alias("stop_at_source"),
        F.to_timestamp(
            "STOP", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("stop_at"),

        blank_to_null("ORGANIZATION").alias("organization_id"),
        blank_to_null("PROVIDER").alias("provider_id"),
        blank_to_null("PAYER").alias("payer_id"),

        blank_to_null("ENCOUNTERCLASS").alias("encounter_class"),
        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        F.col("BASE_ENCOUNTER_COST").alias("base_cost_source"),
        F.col("BASE_ENCOUNTER_COST")
            .cast("decimal(18,2)")
            .alias("base_cost"),

        F.col("TOTAL_CLAIM_COST").alias("total_claim_cost_source"),
        F.col("TOTAL_CLAIM_COST")
            .cast("decimal(18,2)")
            .alias("total_claim_cost"),

        F.col("PAYER_COVERAGE").alias("payer_coverage_source"),
        F.col("PAYER_COVERAGE")
            .cast("decimal(18,2)")
            .alias("payer_coverage"),

        blank_to_null("REASONCODE").alias("reason_code"),
        blank_to_null("REASONDESCRIPTION").alias(
            "reason_description_source"
        ),

        F.lit(None).cast("string").alias("code_system"),
        F.lit("not_supplied").alias("code_system_provenance"),

        *source_lineage_columns()
    )
)

encounters.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.encounters")

**Conditions**

In [0]:
conditions_source = spark.table(f"{STAGED}.conditions")

conditions = (
    conditions_source
    .withColumn(
        "condition_key_sha256",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PATIENT"), F.lit("")),
                F.coalesce(F.col("ENCOUNTER"), F.lit("")),
                F.coalesce(F.col("START"), F.lit("")),
                F.coalesce(F.col("STOP"), F.lit("")),
                F.coalesce(F.col("SYSTEM"), F.lit("")),
                F.coalesce(F.col("CODE"), F.lit(""))
            ),
            256
        )
    )
    .select(
        F.col("condition_key_sha256"),
        blank_to_null("PATIENT").alias("patient_id"),
        blank_to_null("ENCOUNTER").alias("encounter_id"),

        F.col("START").alias("start_date_source"),
        F.to_date("START", "yyyy-MM-dd").alias("start_date"),

        F.col("STOP").alias("stop_date_source"),
        F.to_date("STOP", "yyyy-MM-dd").alias("stop_date"),

        blank_to_null("SYSTEM").alias("code_system"),
        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        *source_lineage_columns()
    )
)

conditions.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.conditions")

**Medications**

In [0]:
medications_source = spark.table(f"{STAGED}.medications")

medications = (
    medications_source
    .withColumn(
        "medication_key_sha256",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PATIENT"), F.lit("")),
                F.coalesce(F.col("ENCOUNTER"), F.lit("")),
                F.coalesce(F.col("START"), F.lit("")),
                F.coalesce(F.col("STOP"), F.lit("")),
                F.coalesce(F.col("CODE"), F.lit("")),
                F.coalesce(F.col("DESCRIPTION"), F.lit(""))
            ),
            256
        )
    )
    .select(
        F.col("medication_key_sha256"),
        blank_to_null("PATIENT").alias("patient_id"),
        blank_to_null("ENCOUNTER").alias("encounter_id"),
        blank_to_null("PAYER").alias("payer_id"),

        F.col("START").alias("start_at_source"),
        F.to_timestamp(
            "START", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("start_at"),

        F.col("STOP").alias("stop_at_source"),
        F.to_timestamp(
            "STOP", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("stop_at"),

        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        F.col("BASE_COST").alias("base_cost_source"),
        F.col("BASE_COST").cast("decimal(18,2)").alias("base_cost"),

        F.col("PAYER_COVERAGE").alias("payer_coverage_source"),
        F.col("PAYER_COVERAGE")
            .cast("decimal(18,2)")
            .alias("payer_coverage"),

        F.col("DISPENSES").alias("dispenses_source"),
        F.col("DISPENSES").cast("integer").alias("dispenses"),

        F.col("TOTALCOST").alias("total_cost_source"),
        F.col("TOTALCOST").cast("decimal(18,2)").alias("total_cost"),

        blank_to_null("REASONCODE").alias("reason_code"),
        blank_to_null("REASONDESCRIPTION").alias(
            "reason_description_source"
        ),

        *source_lineage_columns()
    )
)

medications.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.medications")

**Procedures**

In [0]:
procedures_source = spark.table(f"{STAGED}.procedures")

procedures = (
    procedures_source
    .withColumn(
        "procedure_key_sha256",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PATIENT"), F.lit("")),
                F.coalesce(F.col("ENCOUNTER"), F.lit("")),
                F.coalesce(F.col("START"), F.lit("")),
                F.coalesce(F.col("STOP"), F.lit("")),
                F.coalesce(F.col("SYSTEM"), F.lit("")),
                F.coalesce(F.col("CODE"), F.lit(""))
            ),
            256
        )
    )
    .select(
        F.col("procedure_key_sha256"),
        blank_to_null("PATIENT").alias("patient_id"),
        blank_to_null("ENCOUNTER").alias("encounter_id"),

        F.col("START").alias("start_at_source"),
        F.to_timestamp(
            "START", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("start_at"),

        F.col("STOP").alias("stop_at_source"),
        F.to_timestamp(
            "STOP", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("stop_at"),

        blank_to_null("SYSTEM").alias("code_system"),
        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        F.col("BASE_COST").alias("base_cost_source"),
        F.col("BASE_COST").cast("decimal(18,2)").alias("base_cost"),

        blank_to_null("REASONCODE").alias("reason_code"),
        blank_to_null("REASONDESCRIPTION").alias(
            "reason_description_source"
        ),

        *source_lineage_columns()
    )
)

procedures.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.procedures")

**Observations:**

In [0]:
observations_source = spark.table(f"{STAGED}.observations")

observations = (
    observations_source
    .withColumn(
        "observation_key_sha256",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("PATIENT"), F.lit("")),
                F.coalesce(F.col("ENCOUNTER"), F.lit("")),
                F.coalesce(F.col("DATE"), F.lit("")),
                F.coalesce(F.col("CATEGORY"), F.lit("")),
                F.coalesce(F.col("CODE"), F.lit("")),
                F.coalesce(F.col("VALUE"), F.lit("")),
                F.coalesce(F.col("UNITS"), F.lit(""))
            ),
            256
        )
    )
    .select(
        F.col("observation_key_sha256"),
        blank_to_null("PATIENT").alias("patient_id"),
        blank_to_null("ENCOUNTER").alias("encounter_id"),

        F.col("DATE").alias("observed_at_source"),
        F.to_timestamp(
            "DATE", "yyyy-MM-dd'T'HH:mm:ssX"
        ).alias("observed_at"),

        blank_to_null("CATEGORY").alias("category"),
        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        F.col("VALUE").alias("value_source"),
        blank_to_null("VALUE").alias("value_text"),
        F.expr("try_cast(trim(VALUE) AS DECIMAL(38,10))"
        ).alias("value_numeric"),

        blank_to_null("UNITS").alias("unit_source"),
        blank_to_null("TYPE").alias("value_type_source"),

        *source_lineage_columns()
    )
)

observations.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.observations")

In [0]:
display(
    spark.table(f"{SILVER}.observations")
    .filter(
        F.col("value_numeric").isNull() &
        F.col("value_text").isNotNull()
    )
    .select(
        "description_source",
        "value_source",
        "value_text",
        "value_numeric",
        "unit_source",
        "value_type_source"
    )
    .limit(20)
)

**Care plans**

In [0]:
careplans = (
    spark.table(f"{STAGED}.careplans")
    .select(
        blank_to_null("Id").alias("careplan_id"),
        blank_to_null("PATIENT").alias("patient_id"),
        blank_to_null("ENCOUNTER").alias("encounter_id"),

        F.col("START").alias("start_date_source"),
        F.to_date("START", "yyyy-MM-dd").alias("start_date"),

        F.col("STOP").alias("stop_date_source"),
        F.to_date("STOP", "yyyy-MM-dd").alias("stop_date"),

        blank_to_null("CODE").alias("code"),
        blank_to_null("DESCRIPTION").alias("description_source"),

        blank_to_null("REASONCODE").alias("reason_code"),
        blank_to_null("REASONDESCRIPTION").alias(
            "reason_description_source"
        ),

        *source_lineage_columns()
    )
)

careplans.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.careplans")

**Row-count reconciliation**

In [0]:
datasets = [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "observations",
    "careplans"
]

reconciliation = []

for dataset in datasets:
    staged_count = spark.table(f"{STAGED}.{dataset}").count()
    silver_count = spark.table(f"{SILVER}.{dataset}").count()

    reconciliation.append(
        (dataset, staged_count, silver_count, staged_count == silver_count)
    )

display(
    spark.createDataFrame(
        reconciliation,
        ["dataset", "staged_rows", "silver_rows", "counts_match"]
    )
)

**Relationships**

In [0]:
checks = spark.sql(f"""
SELECT
    (SELECT COUNT(*)
     FROM {SILVER}.encounters e
     LEFT ANTI JOIN {SILVER}.patients p
       ON e.patient_id = p.patient_id
    ) AS orphan_encounters,

    (SELECT COUNT(*)
     FROM {SILVER}.conditions c
     LEFT ANTI JOIN {SILVER}.patients p
       ON c.patient_id = p.patient_id
    ) AS orphan_conditions,

    (SELECT COUNT(*)
     FROM {SILVER}.medications m
     LEFT ANTI JOIN {SILVER}.patients p
       ON m.patient_id = p.patient_id
    ) AS orphan_medications,

    (SELECT COUNT(*)
     FROM {SILVER}.procedures pr
     LEFT ANTI JOIN {SILVER}.patients p
       ON pr.patient_id = p.patient_id
    ) AS orphan_procedures,

    (SELECT COUNT(*)
     FROM {SILVER}.observations o
     LEFT ANTI JOIN {SILVER}.patients p
       ON o.patient_id = p.patient_id
    ) AS orphan_observations,

    (SELECT COUNT(*)
     FROM {SILVER}.observations
     WHERE encounter_id IS NULL
    ) AS encounterless_observations,

    (SELECT COUNT(*)
     FROM {SILVER}.encounters
     WHERE start_at IS NULL OR stop_at IS NULL
    ) AS invalid_encounter_timestamps,

    (SELECT COUNT(*)
     FROM {SILVER}.encounters
     WHERE start_at > stop_at
    ) AS invalid_encounter_intervals
""")

display(checks)